# English → Persian Religious Text Translation

Fine-tuning `facebook/mbart-large-50-many-to-many-mmt` for English-to-Persian
translation of religious/doctrinal text.

**Pipeline:** scrape paired EN/FA pages → strictly align sentence pairs →
fine-tune mBART-50 → evaluate with BLEU.


## 1. Setup

Install dependencies.

In [1]:
!pip install -q requests beautifulsoup4 huggingface_hub evaluate
!pip install -q --upgrade transformers datasets


In [2]:
import transformers
print("transformers version:", transformers.__version__)


transformers version: 5.14.1


## 2. Data Collection

Scrape paired English/Persian pages and produce a strictly-aligned parallel
corpus of sentence pairs.

**Alignment strategy:**
- An entire document is discarded if its English and Persian paragraph
  counts don't match.
- Within an accepted document, each paragraph is discarded unless its
  English and Persian sentence counts match exactly.

This trades corpus size for alignment quality — mismatches are dropped
rather than approximated.


In [3]:
import requests
from bs4 import BeautifulSoup
import json
import re

# ======================================================
# *** CRITICAL: Update your variables here ***
CONTENT_SELECTOR = ".body-block"
OUTPUT_FILE = 'DATASET.jsonl'
PARAGRAPH_TAG = 'p' # Confirmed tag for body text
# ======================================================

# --- HELPER FUNCTIONS ---

def clean_text(text):
    """Normalize text and remove excessive whitespace/newlines."""
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def split_text_into_sentences(text, lang='en'):
    """Segments text into sentences based on punctuation."""
    if not text: return []
    if lang == 'fa':
        # Split on Persian/Arabic end-of-sentence punctuation (. ! ? ؛ ؟)
        sentences = re.split(r'(?<=[.?!؛؟])\s+', text)
    else:
        # Split on standard punctuation (. ! ?)
        sentences = re.split(r'(?<=[.?!])\s+', text)

    return [s.strip() for s in sentences if s.strip()]

def scrape_url_by_elements(url, selector, paragraph_tag='p'):
    """
    Fetches text from a single URL and extracts the content as a list of paragraphs
    by finding all elements matching the paragraph_tag within the main selector.
    """
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')

        # 1. Find the main container element
        main_element = soup.select_one(selector)

        if main_element:
            # 2. Find all paragraph tags (p) within the main element
            paragraph_elements = main_element.find_all(paragraph_tag)

            # 3. Extract, clean, and filter the text from each element
            paragraphs = [clean_text(p.get_text()) for p in paragraph_elements]

            # Filter out any resulting empty strings
            return [p for p in paragraphs if p]

    except Exception as e:
        print(f"❌ Error scraping {url}: {e}")
    return [] # Return an empty list on failure


def scrape_and_process_pair_strict(index, en_url, fa_url, selector, output_file, paragraph_tag='p'):
    """Fetches, aligns strictly by paragraph, and filters by sentence count."""

    # 1. Fetch Texts (now returns lists of paragraphs by HTML element)
    en_paragraphs = scrape_url_by_elements(en_url, selector, paragraph_tag)
    fa_paragraphs = scrape_url_by_elements(fa_url, selector, paragraph_tag)

    print(f"Talk #: {index}")
    print(f"Number of English Paragraphs: {len(en_paragraphs)}")
    print(f"Number of Persian Paragraphs: {len(fa_paragraphs)}")

    if not en_paragraphs or not fa_paragraphs:
        print(f"⚠️ Talk skipped: Could not retrieve or parse paragraphs for: {en_url}. Skipping talk.")
        return

    # 2. STRICT ALIGNMENT CHECK: Paragraph count
    if len(en_paragraphs) != len(fa_paragraphs):
        print(f"⚠️ Talk skipped: Paragraph count mismatch (EN {len(en_paragraphs)} vs FA {len(fa_paragraphs)}) for {en_url}.")
        return

    saved_sentences = 0
    discarded_paragraphs = 0

    with open(output_file, 'a', encoding='utf-8') as f_out:
        for en_para, fa_para in zip(en_paragraphs, fa_paragraphs):

            # 3. Segment Sentences within the aligned paragraphs
            en_sentences = split_text_into_sentences(en_para, lang='en')
            fa_sentences = split_text_into_sentences(fa_para, lang='fa')

            # 4. STRICT SENTENCE FILTER: Check for exact count match AND non-zero count
            if len(en_sentences) == len(fa_sentences) and len(en_sentences) > 0:

                # Save the perfectly aligned sentences
                for en, fa in zip(en_sentences, fa_sentences):
                    entry = {"english": en, "persian": fa}
                    f_out.write(json.dumps(entry, ensure_ascii=False) + '\n')
                    saved_sentences += 1
            else:
                # Discard the entire paragraph (due to mismatch or zero sentences)
                discarded_paragraphs += 1

    print(f"✅ Processed {en_url}. Saved {saved_sentences} pairs. Discarded {discarded_paragraphs} misaligned/empty paragraphs.")


### Run the scraper

Populate `talk_pairs` with your own list of `(english_url, persian_url)`
tuples. For a large list, consider moving this to a separate JSON/CSV file
and loading it here instead of hardcoding it in the notebook.


In [4]:
# --- Example of Batch Usage ---
if __name__ == '__main__':
    # REMEMBER TO INSTALL: !pip install requests beautifulsoup4

    talk_pairs = [
        # Populate with your list of EN/FA URL pairs:
        # ("English_URL_1", "Persian_URL_1"),
        # ("English_URL_2", "Persian_URL_2"),
        # ...

        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/12stevenson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/12stevenson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/13browning?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/13browning?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/14barcellos?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/14barcellos?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/15eyre?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/15eyre?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/17johnson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/17johnson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/16uchtdorf?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/16uchtdorf?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/21rasband?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/21rasband?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/22webb?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/22webb?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/23jaggi?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/23jaggi?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/25gong?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/25gong?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/26cziesla?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/26cziesla?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/27cook?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/27cook?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/31kearon?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/31kearon?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/32dennis?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/32dennis?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/33barlow?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/33barlow?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/34jackson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/34jackson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/35andersen?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/35andersen?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/41holland?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/41holland?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/42evanson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/42evanson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/43soares?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/43soares?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/44johnson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/44johnson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/45christofferson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/45christofferson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/46spannaus?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/46spannaus?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/47eyring?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/47eyring?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/51bednar?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/51bednar?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/52cuvelier?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/52cuvelier?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/53holland?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/53holland?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/54godoy?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/54godoy?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/55renlund?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/55renlund?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/56amos?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/56amos?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/57farias?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/57farias?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/10/58oaks?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/10/58oaks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/13holland?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/13holland?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/14johnson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/14johnson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/15rasband?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/15rasband?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/16cook?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/16cook?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/17gimenez?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/17gimenez?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/18eyring?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/18eyring?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/21andersen?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/21andersen?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/22lund?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/22lund?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/23palmer?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/23palmer?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/24roman?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/24roman?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/25renlund?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/25renlund?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/26boom?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/26boom?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/27uchtdorf?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/27uchtdorf?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/31stevenson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/31stevenson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/32wright?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/32wright?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/33rasband?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/33rasband?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/34vargas?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/34vargas?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/35christofferson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/35christofferson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/41bednar?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/41bednar?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/42shumway?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/42shumway?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/43runia?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/43runia?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/44causse?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/44causse?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/45gong?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/45gong?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/46mccune?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/46mccune?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/47oaks?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/47oaks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/51soares?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/51soares?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/52strong?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/52strong?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/53whiting?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/53whiting?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/54kim?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/54kim?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/55kearon?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/55kearon?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/56tai?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/56tai?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2025/04/57nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2025/04/57nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/12andersen?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/12andersen?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/13freeman?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/13freeman?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/14hirst?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/14hirst?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/15renlund?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/15renlund?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/16homer?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/16homer?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/17casillas?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/17casillas?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/21christofferson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/21christofferson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/22teixeira?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/22teixeira?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/23villar?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/23villar?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/24kearon?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/24kearon?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/25buckner?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/25buckner?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/26goury?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/26goury?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/27cavalcante?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/27cavalcante?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/28soares?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/28soares?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/31gong?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/31gong?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/32yee?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/32yee?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/32yee?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/32yee?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/34alvarado?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/34alvarado?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/35bednar?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/35bednar?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/41holland?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/41holland?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/42browning?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/42browning?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/43hales?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/43hales?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/45budge?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/45budge?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/44stevenson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/44stevenson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/46wilcox?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/46wilcox?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/47eyring?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/47eyring?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/51uchtdorf?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/51uchtdorf?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/52wada?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/52wada?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/53rasband?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/53rasband?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/54cook?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/54cook?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/55alliaud?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/55alliaud?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/56egbo?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/56egbo?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/57nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/57nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/10/57nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/10/57nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/13holland?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/13holland?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/14dennis?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/14dennis?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/15dushku?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/15dushku?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/16soares?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/16soares?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/17gerard?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/17gerard?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/18eyring?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/18eyring?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/21bednar?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/21bednar?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/22de-feo?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/22de-feo?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/23nielson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/23nielson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/24alonso?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/24alonso?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/25gong?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/25gong?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/26nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/26nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/27cook?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/27cook?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/31bowen?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/31bowen?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/32bangerter?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/32bangerter?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/33spannaus?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/33spannaus?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/34carpenter?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/34carpenter?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/35uchtdorf?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/35uchtdorf?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/41rasband?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/41rasband?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/42porter?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/42porter?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/43renlund?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/43renlund?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/44pieper?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/44pieper?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/45kearon?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/45kearon?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/46taylor?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/46taylor?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/47oaks?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/47oaks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/51christofferson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/51christofferson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/52godoy?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/52godoy?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/53stevenson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/53stevenson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/54held?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/54held?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/55andersen?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/55andersen?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/56pace?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/56pace?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2024/04/57nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2024/04/57nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/11bednar?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/11bednar?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/12wright?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/12wright?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/13daines?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/13daines?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/14godoy?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/14godoy?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/15christofferson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/15christofferson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/16ardern?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/16ardern?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/17oaks?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/17oaks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/22andersen?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/22andersen?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/23newman?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/23newman?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/24costa?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/24costa?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/25stevenson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/25stevenson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/26choi?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/26choi?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/27phillips?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/27phillips?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/28rasband?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/28rasband?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/31sabin?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/31sabin?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/32koch?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/32koch?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/33runia?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/33runia?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/33runia?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/33runia?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/34soares?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/34soares?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/41ballard?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/41ballard?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/42freeman?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/42freeman?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/43parrella?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/43parrella?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/44cook?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/44cook?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/45uchtdorf?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/45uchtdorf?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/46waddell?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/46waddell?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/47eyring?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/47eyring?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/57renlund?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/57renlund?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/52pingree?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/52pingree?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/53cordon?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/53cordon?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/55esplin?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/55esplin?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/54gong?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/54gong?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/56giraud-carrier?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/56giraud-carrier?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/10/51nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/10/51nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/11stevenson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/11stevenson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/12cordon?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/12cordon?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/13cook?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/13cook?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/14gong?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/14gong?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/15cook?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/15cook?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/16haynie?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/16haynie?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/17eyring?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/17eyring?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/23renlund?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/23renlund?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/24meurs?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/24meurs?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/25bennett?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/25bennett?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/26christensen?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/26christensen?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/27schmutz?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/27schmutz?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/28de-hoyos?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/28de-hoyos?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/29uchtdorf?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/29uchtdorf?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/31bragg?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/31bragg?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/32camargo?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/32camargo?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/33nattress?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/33nattress?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/34uceda?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/34uceda?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/41christofferson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/41christofferson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/42johnson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/42johnson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/43soares?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/43soares?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/44yamashita?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/44yamashita?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/45andersen?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/45andersen?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/46duncan?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/46duncan?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2023/04/47nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2023/04/47nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/18oaks?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/18oaks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/12uchtdorf?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/12uchtdorf?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/13browning?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/13browning?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/14renlund?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/14renlund?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/15pino?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/15pino?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/16montoya?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/16montoya?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/17rasband?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/17rasband?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/19nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/19nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/22ballard?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/22ballard?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/23yee?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/23yee?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/24johnson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/24johnson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/25soares?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/25soares?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/26mcconkie?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/26mcconkie?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/27zeballos?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/27zeballos?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/28christofferson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/28christofferson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/31causse?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/31causse?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/32craig?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/32craig?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/33pearson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/33pearson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/34silva?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/34silva?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/35andersen?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/35andersen?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/41holland?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/41holland?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/42dennis?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/42dennis?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/43gong?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/43gong?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/44sitati?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/44sitati?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/45lund?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/45lund?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/46bednar?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/46bednar?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/47nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/47nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/51eyring?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/51eyring?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/52olsen?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/52olsen?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/53schmitt?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/53schmitt?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/54eddy?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/54eddy?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/55stevenson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/55stevenson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/56morrison?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/56morrison?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/57cook?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/57cook?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/10/58nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/10/58nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/11nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/11nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/12ballard?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/12ballard?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/13aburto?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/13aburto?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/14bednar?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/14bednar?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/15andersen?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/15andersen?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/16gavarret?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/16gavarret?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/17kacher?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/17kacher?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/18eyring?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/18eyring?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/23holland?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/23holland?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/24kearon?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/24kearon?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/25aidukaitis?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/25aidukaitis?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/26gong?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/26gong?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/27ochoa?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/27ochoa?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/28hamilton?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/28hamilton?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/29cook?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/29cook?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/31oaks?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/31oaks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/32porter?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/32porter?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/33craven?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/33craven?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/35bingham?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/35bingham?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/36renlund?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/36renlund?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/41christofferson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/41christofferson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/42wright?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/42wright?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/43stevenson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/43stevenson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/44ringwood?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/44ringwood?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/45rasband?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/45rasband?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/46martinez?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/46martinez?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/47nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/47nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/51oaks?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/51oaks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/52ojediran?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/52ojediran?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/53klebingat?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/53klebingat?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/54pace?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/54pace?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/55soares?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/55soares?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/56funk?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/56funk?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/57uchtdorf?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/57uchtdorf?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2022/04/58nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2022/04/58nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/11nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/11nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/12holland?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/12holland?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/13cordon?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/13cordon?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/14soares?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/14soares?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/15christofferson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/15christofferson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/16gilbert?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/16gilbert?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/17giuffra?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/17giuffra?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/18oaks?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/18oaks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/22bednar?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/22bednar?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/23schmeil?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/23schmeil?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/24porter?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/24porter?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/25kopischke?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/25kopischke?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/26rasband?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/26rasband?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/27golden?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/27golden?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/28villanueva?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/28villanueva?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/29stevenson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/29stevenson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/31ballard?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/31ballard?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/32eubank?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/32eubank?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/33nielson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/33nielson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/34valenzuela?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/34valenzuela?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/35wilcox?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/35wilcox?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/36kyungu?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/36kyungu?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/37nash?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/37nash?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/38eyring?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/38eyring?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/41uchtdorf?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/41uchtdorf?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/42johnson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/42johnson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/43renlund?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/43renlund?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/44sikahema?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/44sikahema?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/46cook?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/46cook?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/47nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/47nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/51gong?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/51gong?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/52budge?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/52budge?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/53perkins?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/53perkins?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/54dunn?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/54dunn?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/55douglas?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/55douglas?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/56revillo?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/56revillo?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/57meredith?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/57meredith?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/58andersen?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/58andersen?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/10/59nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/10/59nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/11nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/11nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/12uchtdorf?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/12uchtdorf?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/13jones?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/13jones?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/14newman?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/14newman?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/15stevenson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/15stevenson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/16gong?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/16gong?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/17eyring?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/17eyring?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/23holland?lang=eng","https://www.churchofjesuschrist.org/study/general-conference/2021/04/23holland?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/24becerra?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/24becerra?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/25renlund?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/25renlund?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/26andersen?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/26andersen?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/27mutombo?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/27mutombo?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/28ballard?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/28ballard?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/31cook?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/31cook?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/32corbitt?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/32corbitt?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/33nielsen?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/33nielsen?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/34eyring?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/34eyring?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/35oaks?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/35oaks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/36nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/36nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/41soares?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/41soares?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/42aburto?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/42aburto?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/43palmer?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/43palmer?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/44dube?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/44dube?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/45teixeira?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/45teixeira?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/46wakolo?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/46wakolo?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/47wong?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/47wong?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/48teh?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/48teh?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/49nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/49nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/51oaks?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/51oaks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/52rasband?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/52rasband?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/53dyches?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/53dyches?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/54christofferson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/54christofferson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/55walker?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/55walker?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/56bednar?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/56bednar?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2021/04/57nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2021/04/57nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/11nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/11nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/12bednar?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/12bednar?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/13whiting?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/13whiting?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/14craig?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/14craig?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/15cook?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/15cook?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/16rasband?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/16rasband?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/17oaks?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/17oaks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/22christofferson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/22christofferson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/23lund?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/23lund?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/24gong?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/24gong?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/25waddell?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/25waddell?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/26holland?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/26holland?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/27jackson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/27jackson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/28uchtdorf?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/28uchtdorf?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/31eubank?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/31eubank?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/32craven?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/32craven?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/34franco?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/34franco?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/35eyring?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/35eyring?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/36oaks?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/36oaks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/37nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/37nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/41ballard?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/41ballard?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/42harkness?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/42harkness?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/43soares?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/43soares?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/44godoy?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/44godoy?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/45andersen?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/45andersen?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/46nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/46nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/51eyring?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/51eyring?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/52jaggi?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/52jaggi?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/53stevenson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/53stevenson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/54camargo?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/54camargo?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/55renlund?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/55renlund?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/56johnson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/56johnson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/57holland?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/57holland?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/10/58nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/10/58nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/11nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/11nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/12ballard?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/12ballard?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/13rasband?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/13rasband?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/14jones?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/14jones?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/15andersen?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/15andersen?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/16holmes?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/16holmes?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/17eyring?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/17eyring?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/23soares?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/23soares?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/24mccune?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/24mccune?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/25causse?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/25causse?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/26renlund?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/26renlund?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/27tai?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/27tai?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/28stevenson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/28stevenson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/31gong?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/31gong?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/32alvarez?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/32alvarez?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/33petelo?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/33petelo?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/34bingham?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/34bingham?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/35eyring?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/35eyring?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/36oaks?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/36oaks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/37nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/37nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/41rasband?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/41rasband?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/42cordon?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/42cordon?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/43holland?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/43holland?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/44bednar?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/44bednar?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/45nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/45nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/46nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/46nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/51oaks?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/51oaks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/52cook?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/52cook?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/53gimenez?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/53gimenez?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/54uchtdorf?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/54uchtdorf?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/55clayton?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/55clayton?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/56christofferson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/56christofferson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2020/04/57nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2020/04/57nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/11holland?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/11holland?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/12vinson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/12vinson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/13owen?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/13owen?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/14christofferson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/14christofferson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/15craig?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/15craig?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/16renlund?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/16renlund?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/17oaks?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/17oaks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/22bednar?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/22bednar?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/23alliaud?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/23alliaud?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/24nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/24nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/25cook?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/25cook?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/26pace?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/26pace?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/27budge?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/27budge?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/28alvarado?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/28alvarado?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/29rasband?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/29rasband?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/31aburto?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/31aburto?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/32harkness?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/32harkness?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/33cordon?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/33cordon?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/34eyring?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/34eyring?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/35oaks?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/35oaks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/36nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/36nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/41gong?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/41gong?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/42franco?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/42franco?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/43uchtdorf?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/43uchtdorf?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/44gonzalez?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/44gonzalez?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/45stevenson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/45stevenson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/46nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/46nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/51eyring?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/51eyring?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/52boom?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/52boom?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/53ballard?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/53ballard?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/54johnson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/54johnson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/55soares?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/55soares?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/56andersen?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/56andersen?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/10/57nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/10/57nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/11soares?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/11soares?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/12craven?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/12craven?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/13hales?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/13hales?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/14uchtdorf?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/14uchtdorf?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/15waddell?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/15waddell?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/16eyring?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/16eyring?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/23ballard?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/23ballard?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/24held?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/24held?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/25andersen?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/25andersen?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/26wada?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/26wada?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/27homer?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/27homer?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/28holland?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/28holland?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/31stevenson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/31stevenson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/32cook?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/32cook?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/33clark?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/33clark?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/34eyring?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/34eyring?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/35oaks?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/35oaks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/36nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/36nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/41renlund?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/41renlund?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/42eubank?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/42eubank?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/43cook?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/43cook?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/44christofferson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/44christofferson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/45callister?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/45callister?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/46nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/46nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/51oaks?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/51oaks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/52villar?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/52villar?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/53gong?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/53gong?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/54bednar?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/54bednar?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/55mckay?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/55mckay?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/56rasband?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/56rasband?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2019/04/57nelson?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2019/04/57nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/opening-remarks?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/opening-remarks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/deep-and-lasting-conversion-to-heavenly-father-and-the-lord-jesus-christ?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/deep-and-lasting-conversion-to-heavenly-father-and-the-lord-jesus-christ?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/lift-up-your-head-and-rejoice?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/lift-up-your-head-and-rejoice?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/laying-the-foundation-of-a-great-work?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/laying-the-foundation-of-a-great-work?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/be-not-troubled?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/be-not-troubled?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/gather-together-in-one-all-things-in-christ?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/gather-together-in-one-all-things-in-christ?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/truth-and-the-plan?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/truth-and-the-plan?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/firm-and-steadfast-in-the-faith-of-christ?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/firm-and-steadfast-in-the-faith-of-christ?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/come-listen-to-a-prophets-voice?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/come-listen-to-a-prophets-voice?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/one-in-christ?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/one-in-christ?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/our-campfire-of-faith?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/our-campfire-of-faith?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/all-must-take-upon-them-the-name-given-of-the-father?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/all-must-take-upon-them-the-name-given-of-the-father?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/believe-love-do?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/believe-love-do?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/for-him?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/for-him?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/divine-discontent?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/divine-discontent?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/the-joy-of-unselfish-service?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/the-joy-of-unselfish-service?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/women-and-gospel-learning-in-the-home?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/women-and-gospel-learning-in-the-home?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/parents-and-children?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/parents-and-children?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/sisters-participation-in-the-gathering-of-israel?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/sisters-participation-in-the-gathering-of-israel?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/the-vision-of-the-redemption-of-the-dead?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/the-vision-of-the-redemption-of-the-dead?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/becoming-a-shepherd?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/becoming-a-shepherd?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/the-ministry-of-reconciliation?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/the-ministry-of-reconciliation?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/the-role-of-the-book-of-mormon-in-conversion?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/the-role-of-the-book-of-mormon-in-conversion?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/wounded?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/wounded?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/the-correct-name-of-the-church?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/the-correct-name-of-the-church?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/try-try-try?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/try-try-try?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/the-father?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/the-father?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/taking-upon-ourselves-the-name-of-jesus-christ?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/taking-upon-ourselves-the-name-of-jesus-christ?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/wilt-thou-be-made-whole?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/wilt-thou-be-made-whole?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/choose-you-this-day?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/choose-you-this-day?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/now-is-the-time?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/now-is-the-time?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/shepherding-souls?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/shepherding-souls?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/10/becoming-exemplary-latter-day-saints?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/10/becoming-exemplary-latter-day-saints?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/solemn-assembly?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/solemn-assembly?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/precious-gifts-from-god?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/precious-gifts-from-god?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/am-i-a-child-of-god?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/am-i-a-child-of-god?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/even-as-christ-forgives-you-so-also-do-ye?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/even-as-christ-forgives-you-so-also-do-ye?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/the-heart-of-a-prophet?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/the-heart-of-a-prophet?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/until-seventy-times-seven?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/until-seventy-times-seven?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/the-prophet-of-god?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/the-prophet-of-god?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/meek-and-lowly-of-heart?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/meek-and-lowly-of-heart?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/one-more-day?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/one-more-day?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/young-women-in-the-work?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/young-women-in-the-work?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/saving-ordinances-will-bring-us-marvelous-light?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/saving-ordinances-will-bring-us-marvelous-light?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/teaching-in-the-home-a-joyful-and-sacred-responsibility?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/teaching-in-the-home-a-joyful-and-sacred-responsibility?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/family-history-and-temple-work-sealing-and-healing?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/family-history-and-temple-work-sealing-and-healing?lang=pes"),
        {"https://www.churchofjesuschrist.org/study/general-conference/2018/04/what-every-aaronic-priesthood-holder-needs-to-understand?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/what-every-aaronic-priesthood-holder-needs-to-understand?lang=pes"},
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/introductory-remarks?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/introductory-remarks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/the-elders-quorum?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/the-elders-quorum?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/behold-a-royal-army?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/behold-a-royal-army?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/inspired-ministering?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/inspired-ministering?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/the-powers-of-the-priesthood?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/the-powers-of-the-priesthood?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/ministering-with-the-power-and-authority-of-god?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/ministering-with-the-power-and-authority-of-god?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/take-the-holy-spirit-as-your-guide?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/take-the-holy-spirit-as-your-guide?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/with-one-accord?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/with-one-accord?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/pure-love-the-true-sign-of-every-true-disciple-of-jesus-christ?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/pure-love-the-true-sign-of-every-true-disciple-of-jesus-christ?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/he-that-shall-endure-unto-the-end-the-same-shall-be-saved?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/he-that-shall-endure-unto-the-end-the-same-shall-be-saved?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/his-spirit-to-be-with-you?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/his-spirit-to-be-with-you?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/small-and-simple-things?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/small-and-simple-things?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/revelation-for-the-church-revelation-for-our-lives?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/revelation-for-the-church-revelation-for-our-lives?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/christ-the-lord-is-risen-today?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/christ-the-lord-is-risen-today?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/prophets-speak-by-the-power-of-the-holy-spirit?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/prophets-speak-by-the-power-of-the-holy-spirit?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/ministering?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/ministering?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/be-with-and-strengthen-them?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/be-with-and-strengthen-them?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/ministering-as-the-savior-does?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/ministering-as-the-savior-does?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/behold-the-man?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/behold-the-man?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/it-is-all-about-people?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/it-is-all-about-people?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/prepare-to-meet-god?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/prepare-to-meet-god?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2018/04/let-us-all-press-on?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2018/04/let-us-all-press-on?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/a-yearning-for-home?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/a-yearning-for-home?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/the-needs-before-us?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/the-needs-before-us?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/the-plan-and-the-proclamation?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/the-plan-and-the-proclamation?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/i-have-a-work-for-thee?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/i-have-a-work-for-thee?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/the-living-bread-which-came-down-from-heaven?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/the-living-bread-which-came-down-from-heaven?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/be-ye-therefore-perfect-eventually?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/be-ye-therefore-perfect-eventually?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/spiritual-eclipse?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/spiritual-eclipse?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/repentance-is-always-positive?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/repentance-is-always-positive?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/the-eternal-everyday?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/the-eternal-everyday?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/by-divine-design?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/by-divine-design?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/the-heart-of-the-widow?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/the-heart-of-the-widow?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/the-book-of-mormon-what-would-your-life-be-like-without-it?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/the-book-of-mormon-what-would-your-life-be-like-without-it?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/the-priesthood-and-the-saviors-atoning-power?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/the-priesthood-and-the-saviors-atoning-power?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/the-truth-of-all-things?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/the-truth-of-all-things?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/earning-the-trust-of-the-lord-and-your-family?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/earning-the-trust-of-the-lord-and-your-family?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/bearers-of-heavenly-light?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/bearers-of-heavenly-light?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/the-lord-leads-his-church?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/the-lord-leads-his-church?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/that-your-joy-might-be-full?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/that-your-joy-might-be-full?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/has-the-day-of-miracles-ceased?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/has-the-day-of-miracles-ceased?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/exceeding-great-and-precious-promises?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/exceeding-great-and-precious-promises?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/turn-to-the-lord?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/turn-to-the-lord?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/lord-wilt-thou-cause-that-my-eyes-may-be-opened?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/lord-wilt-thou-cause-that-my-eyes-may-be-opened?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/fear-not-to-do-good?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/fear-not-to-do-good?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/the-trek-continues?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/the-trek-continues?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/gods-compelling-witness-the-book-of-mormon?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/gods-compelling-witness-the-book-of-mormon?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/apart-but-still-one?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/apart-but-still-one?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/do-we-trust-him-hard-is-good?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/do-we-trust-him-hard-is-good?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/essential-truths-our-need-to-act?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/essential-truths-our-need-to-act?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/seek-ye-out-of-the-best-books?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/seek-ye-out-of-the-best-books?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/love-one-another-as-he-has-loved-us?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/love-one-another-as-he-has-loved-us?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/10/the-voice-of-the-lord?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/10/the-voice-of-the-lord?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/gathering-the-family-of-god?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/gathering-the-family-of-god?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/his-daily-guiding-hand?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/his-daily-guiding-hand?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/our-fathers-glorious-plan?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/our-fathers-glorious-plan?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/our-good-shepherd?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/our-good-shepherd?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/confide-in-god-unwaveringly?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/confide-in-god-unwaveringly?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/brighter-and-brighter-until-the-perfect-day?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/brighter-and-brighter-until-the-perfect-day?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/drawing-the-power-of-jesus-christ-into-our-lives?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/drawing-the-power-of-jesus-christ-into-our-lives?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/becoming-a-disciple-of-our-lord-jesus-christ?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/becoming-a-disciple-of-our-lord-jesus-christ?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/songs-sung-and-unsung?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/songs-sung-and-unsung?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/stand-up-inside-and-be-all-in?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/stand-up-inside-and-be-all-in?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/the-language-of-the-gospel?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/the-language-of-the-gospel?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/overcoming-the-world?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/overcoming-the-world?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/return-and-receive?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/return-and-receive?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/kindness-charity-and-love?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/kindness-charity-and-love?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/called-to-the-work?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/called-to-the-work?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/prepare-the-way?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/prepare-the-way?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/the-greatest-among-you?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/the-greatest-among-you?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/walk-with-me?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/walk-with-me?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/the-power-of-the-book-of-mormon?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/the-power-of-the-book-of-mormon?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/a-sin-resistant-generation?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/a-sin-resistant-generation?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/dont-look-around-look-up?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/dont-look-around-look-up?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/let-the-holy-spirit-guide?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/let-the-holy-spirit-guide?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/whatsoever-he-saith-unto-you-do-it?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/whatsoever-he-saith-unto-you-do-it?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/the-godhead-and-the-plan-of-salvation?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/the-godhead-and-the-plan-of-salvation?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/perfect-love-casteth-out-fear?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/perfect-love-casteth-out-fear?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/the-voice-of-warning?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/the-voice-of-warning?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/to-the-friends-and-investigators-of-the-church?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/to-the-friends-and-investigators-of-the-church?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/then-jesus-beholding-him-loved-him?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/then-jesus-beholding-him-loved-him?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/how-does-the-holy-ghost-help-you?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/how-does-the-holy-ghost-help-you?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/and-this-is-life-eternal?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/and-this-is-life-eternal?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/that-our-light-may-be-a-standard-for-the-nations?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/that-our-light-may-be-a-standard-for-the-nations?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2017/04/foundations-of-faith?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2017/04/foundations-of-faith?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/o-how-great-the-plan-of-our-god?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/o-how-great-the-plan-of-our-god?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/come-follow-me-by-practicing-christian-love-and-service?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/come-follow-me-by-practicing-christian-love-and-service?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/the-souls-sincere-desire?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/the-souls-sincere-desire?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/a-choice-seer-will-i-raise-up?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/a-choice-seer-will-i-raise-up?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/the-lord-jesus-christ-teaches-us-to-pray?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/the-lord-jesus-christ-teaches-us-to-pray?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/am-i-good-enough-will-i-make-it?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/am-i-good-enough-will-i-make-it?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/a-witness-of-god?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/a-witness-of-god?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/valiant-in-the-testimony-of-jesus?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/valiant-in-the-testimony-of-jesus?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/look-to-the-book-look-to-the-lord?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/look-to-the-book-look-to-the-lord?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/abide-in-my-love?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/abide-in-my-love?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/for-our-spiritual-development-and-learning?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/for-our-spiritual-development-and-learning?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/be-ambitious-for-christ?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/be-ambitious-for-christ?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/sharing-the-restored-gospel?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/sharing-the-restored-gospel?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/the-perfect-path-to-happiness?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/the-perfect-path-to-happiness?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/joy-and-spiritual-survival?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/joy-and-spiritual-survival?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/the-sacrament-can-help-us-become-holy?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/the-sacrament-can-help-us-become-holy?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/the-great-plan-of-redemption?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/the-great-plan-of-redemption?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/to-whom-shall-we-go?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/to-whom-shall-we-go?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/the-blessings-of-worship?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/the-blessings-of-worship?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/the-righteous-judge?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/the-righteous-judge?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/gratitude-on-the-sabbath-day?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/gratitude-on-the-sabbath-day?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/if-ye-had-known-me?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/if-ye-had-known-me?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/the-doctrine-of-christ?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/the-doctrine-of-christ?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/serve?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/serve?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/lest-thou-forget?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/lest-thou-forget?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/god-shall-wipe-away-all-tears?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/god-shall-wipe-away-all-tears?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/no-greater-joy-than-to-know-that-they-know?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/no-greater-joy-than-to-know-that-they-know?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/10/repentance-a-joyful-choice?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/10/repentance-a-joyful-choice?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/04/a-childs-guiding-gift?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/04/a-childs-guiding-gift?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/04/i-am-a-child-of-god?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/04/i-am-a-child-of-god?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/04/where-are-the-keys-and-authority-of-the-priesthood?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/04/where-are-the-keys-and-authority-of-the-priesthood?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/04/the-healing-ointment-of-forgiveness?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/04/the-healing-ointment-of-forgiveness?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/04/be-thou-humble?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/04/be-thou-humble?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/04/that-i-might-draw-all-men-unto-me?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/04/that-i-might-draw-all-men-unto-me?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/04/standing-with-the-leaders-of-the-church?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/04/standing-with-the-leaders-of-the-church?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/04/to-the-rescue-we-can-do-it?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/04/to-the-rescue-we-can-do-it?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/04/the-sacred-place-of-restoration?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/04/the-sacred-place-of-restoration?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/04/always-retain-a-remission-of-your-sins?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/04/always-retain-a-remission-of-your-sins?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/04/family-councils?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/04/family-councils?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/04/choices?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/04/choices?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/04/do-i-believe?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/04/do-i-believe?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/04/a-pattern-for-peace?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/04/a-pattern-for-peace?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/04/fathers?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/04/fathers?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/04/he-will-place-you-on-his-shoulders-and-carry-you-home?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/04/he-will-place-you-on-his-shoulders-and-carry-you-home?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/04/the-holy-ghost?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/04/the-holy-ghost?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/04/always-remember-him?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/04/always-remember-him?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2016/04/opposition-in-all-things?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2016/04/opposition-in-all-things?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/it-works-wonderfully?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/it-works-wonderfully?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/god-is-at-the-helm?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/god-is-at-the-helm?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/the-joy-of-living-a-christ-centered-life?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/the-joy-of-living-a-christ-centered-life?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/yielding-our-hearts-to-god?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/yielding-our-hearts-to-god?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/the-pleasing-word-of-god?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/the-pleasing-word-of-god?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/shipshape-and-bristol-fashion-be-temple-worthy-in-good-times-and-bad-times?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/shipshape-and-bristol-fashion-be-temple-worthy-in-good-times-and-bad-times?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/meeting-the-challenges-of-todays-world?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/meeting-the-challenges-of-todays-world?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/behold-thy-mother?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/behold-thy-mother?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/its-never-too-early-and-its-never-too-late?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/its-never-too-early-and-its-never-too-late?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/tested-and-tempted-but-helped?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/tested-and-tempted-but-helped?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/choose-the-light?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/choose-the-light?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/i-stand-all-amazed?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/i-stand-all-amazed?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/plain-and-precious-truths?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/plain-and-precious-truths?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/through-gods-eyes?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/through-gods-eyes?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/a-plea-to-my-sisters?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/a-plea-to-my-sisters?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/let-the-clarion-trumpet-sound?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/let-the-clarion-trumpet-sound?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/that-they-do-always-remember-him?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/that-they-do-always-remember-him?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/the-holy-ghost-as-your-companion?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/the-holy-ghost-as-your-companion?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/why-the-church?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/why-the-church?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/my-heart-pondereth-them-continually?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/my-heart-pondereth-them-continually?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/blessed-and-happy-are-those-who-keep-the-commandments-of-god?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/blessed-and-happy-are-those-who-keep-the-commandments-of-god?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/if-ye-love-me-keep-my-commandments?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/if-ye-love-me-keep-my-commandments?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/remembering-in-whom-we-have-trusted?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/remembering-in-whom-we-have-trusted?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/eyes-to-see-and-ears-to-hear?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/eyes-to-see-and-ears-to-hear?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/hold-on-thy-way?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/hold-on-thy-way?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/10/chosen-to-bear-testimony-of-my-name?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/10/chosen-to-bear-testimony-of-my-name?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/is-not-this-the-fast-that-i-have-chosen?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/is-not-this-the-fast-that-i-have-chosen?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/the-plan-of-happiness?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/the-plan-of-happiness?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/well-ascend-together?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/well-ascend-together?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/the-parable-of-the-sower?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/the-parable-of-the-sower?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/choose-to-believe?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/choose-to-believe?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/why-marriage-and-family-matter-everywhere-in-the-world?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/why-marriage-and-family-matter-everywhere-in-the-world?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/therefore-they-hushed-their-fears?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/therefore-they-hushed-their-fears?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/why-marriage-why-family?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/why-marriage-why-family?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/the-music-of-the-gospel?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/the-music-of-the-gospel?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/latter-day-saints-keep-on-trying?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/latter-day-saints-keep-on-trying?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/truly-good-and-without-guile?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/truly-good-and-without-guile?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/the-lord-is-my-light?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/the-lord-is-my-light?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/blessings-of-the-temple?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/blessings-of-the-temple?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/returning-to-faith?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/returning-to-faith?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/seeking-the-lord?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/seeking-the-lord?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/is-it-still-wonderful-to-you?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/is-it-still-wonderful-to-you?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/waiting-for-the-prodigal?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/waiting-for-the-prodigal?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/where-justice-love-and-mercy-meet?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/where-justice-love-and-mercy-meet?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/the-gift-of-grace?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/the-gift-of-grace?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/preserving-agency-protecting-religious-freedom?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/preserving-agency-protecting-religious-freedom?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/stay-by-the-tree?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/stay-by-the-tree?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/the-eternal-perspective-of-the-gospel?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/the-eternal-perspective-of-the-gospel?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/thy-kingdom-come?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/thy-kingdom-come?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/if-you-will-be-responsible?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/if-you-will-be-responsible?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/be-fruitful-multiply-and-subdue-the-earth?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/be-fruitful-multiply-and-subdue-the-earth?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2015/04/the-sabbath-is-a-delight?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2015/04/the-sabbath-is-a-delight?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/welcome-to-conference?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/welcome-to-conference?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/the-reason-for-our-hope?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/the-reason-for-our-hope?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/which-way-do-you-face?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/which-way-do-you-face?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/the-sacrament-a-renewal-for-the-soul?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/the-sacrament-a-renewal-for-the-soul?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/rescue-in-unity?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/rescue-in-unity?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/free-forever-to-act-for-themselves?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/free-forever-to-act-for-themselves?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/receiving-a-testimony-of-light-and-truth?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/receiving-a-testimony-of-light-and-truth?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/loving-others-and-living-with-differences?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/loving-others-and-living-with-differences?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/joseph-smith?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/joseph-smith?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/parents-the-prime-gospel-teachers-of-their-children?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/parents-the-prime-gospel-teachers-of-their-children?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/approaching-the-throne-of-god-with-confidence?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/approaching-the-throne-of-god-with-confidence?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/yes-lord-i-will-follow-thee?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/yes-lord-i-will-follow-thee?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/are-we-not-all-beggars?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/are-we-not-all-beggars?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/finding-lasting-peace-and-building-eternal-families?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/finding-lasting-peace-and-building-eternal-families?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/continuing-revelation?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/continuing-revelation?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/sustaining-the-prophets?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/sustaining-the-prophets?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/live-according-to-the-words-of-the-prophets?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/live-according-to-the-words-of-the-prophets?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/eternal-life-to-know-our-heavenly-father-and-his-son-jesus-christ?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/eternal-life-to-know-our-heavenly-father-and-his-son-jesus-christ?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/the-sacrament-and-the-atonement?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/the-sacrament-and-the-atonement?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/ponder-the-path-of-thy-feet?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/ponder-the-path-of-thy-feet?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/stay-in-the-boat-and-hold-on?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/stay-in-the-boat-and-hold-on?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/make-the-exercise-of-faith-your-first-priority?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/make-the-exercise-of-faith-your-first-priority?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/the-lord-has-a-plan-for-us?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/the-lord-has-a-plan-for-us?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/the-book?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/the-book?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/our-personal-ministries?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/our-personal-ministries?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/trifle-not-with-sacred-things?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/trifle-not-with-sacred-things?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/come-and-see?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/come-and-see?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/10/until-we-meet-again?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/10/until-we-meet-again?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/04/welcome-to-conference?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/04/welcome-to-conference?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/04/christ-the-redeemer?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/04/christ-the-redeemer?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/04/spiritual-whirlwinds?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/04/spiritual-whirlwinds?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/04/a-priceless-heritage-of-hope?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/04/a-priceless-heritage-of-hope?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/04/let-your-faith-show?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/04/let-your-faith-show?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/04/lets-not-take-the-wrong-way?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/04/lets-not-take-the-wrong-way?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/04/roots-and-branches?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/04/roots-and-branches?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/general-conference/2014/04/your-four-minutes?lang=eng", "https://www.churchofjesuschrist.org/study/general-conference/2014/04/your-four-minutes?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/teaching-in-the-saviors-way-2022/04-part-1/05-teach-about-jesus-christ?lang=eng", "https://www.churchofjesuschrist.org/study/manual/teaching-in-the-saviors-way-2022/04-part-1/05-teach-about-jesus-christ?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/teaching-in-the-saviors-way-2022/04-part-1/06-help-learners-come-unto-christ?lang=eng", "https://www.churchofjesuschrist.org/study/manual/teaching-in-the-saviors-way-2022/04-part-1/06-help-learners-come-unto-christ?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/teaching-in-the-saviors-way-2022/07-part-2/08-love-those-you-teach?lang=eng", "https://www.churchofjesuschrist.org/study/manual/teaching-in-the-saviors-way-2022/07-part-2/08-love-those-you-teach?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/teaching-in-the-saviors-way-2022/07-part-2/09-teach-by-the-spirit?lang=eng", "https://www.churchofjesuschrist.org/study/manual/teaching-in-the-saviors-way-2022/07-part-2/09-teach-by-the-spirit?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/teaching-in-the-saviors-way-2022/07-part-2/10-teach-the-doctrine?lang=eng", "https://www.churchofjesuschrist.org/study/manual/teaching-in-the-saviors-way-2022/07-part-2/10-teach-the-doctrine?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/teaching-in-the-saviors-way-2022/07-part-2/11-invite-diligent-learning?lang=eng", "https://www.churchofjesuschrist.org/study/manual/teaching-in-the-saviors-way-2022/07-part-2/11-invite-diligent-learning?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/teaching-in-the-saviors-way-2022/12-part-3/13-suggestions-for-a-variety-of-teaching?lang=eng", "https://www.churchofjesuschrist.org/study/manual/teaching-in-the-saviors-way-2022/12-part-3/13-suggestions-for-a-variety-of-teaching?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/teaching-in-the-saviors-way-2022/12-part-3/17-teacher-council-meetings-for-parents-and-teachers?lang=eng", "https://www.churchofjesuschrist.org/study/manual/teaching-in-the-saviors-way-2022/12-part-3/17-teacher-council-meetings-for-parents-and-teachers?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/preach-my-gospel-2023/04-chapter-3/06-chapter-3-intro?lang=eng", "https://www.churchofjesuschrist.org/study/manual/preach-my-gospel-2023/04-chapter-3/06-chapter-3-intro?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/preach-my-gospel-2023/04-chapter-3/07-chapter-3-invite?lang=eng", "https://www.churchofjesuschrist.org/study/manual/preach-my-gospel-2023/04-chapter-3/07-chapter-3-invite?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/preach-my-gospel-2023/04-chapter-3/08-chapter-3-lesson-1?lang=eng", "https://www.churchofjesuschrist.org/study/manual/preach-my-gospel-2023/04-chapter-3/08-chapter-3-lesson-1?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/preach-my-gospel-2023/04-chapter-3/09-chapter-3-lesson-2?lang=eng", "https://www.churchofjesuschrist.org/study/manual/preach-my-gospel-2023/04-chapter-3/09-chapter-3-lesson-2?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/preach-my-gospel-2023/04-chapter-3/10-chapter-3-lesson-3?lang=eng", "https://www.churchofjesuschrist.org/study/manual/preach-my-gospel-2023/04-chapter-3/10-chapter-3-lesson-3?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/preach-my-gospel-2023/04-chapter-3/11-chapter-3-lesson-4?lang=eng", "https://www.churchofjesuschrist.org/study/manual/preach-my-gospel-2023/04-chapter-3/11-chapter-3-lesson-4?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/004-intro?lang=eng", "https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/004-intro?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/01-mutual-respect?lang=eng", "https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/01-mutual-respect?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/02-profession-of-faith-in-god?lang=eng", "https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/02-profession-of-faith-in-god?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/03-the-posterity-of-abraham?lang=eng", "https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/03-the-posterity-of-abraham?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/04-prophets?lang=eng", "https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/04-prophets?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/05-jesus-christ?lang=eng", "https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/05-jesus-christ?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/06-scriptures?lang=eng", "https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/06-scriptures?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/07-prayer?lang=eng", "https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/07-prayer?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/08-helping-those-in-need?lang=eng", "https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/08-helping-those-in-need?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/09-fasting?lang=eng", "https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/09-fasting?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/10-physical-health?lang=eng", "https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/10-physical-health?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/11-chastity?lang=eng", "https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/11-chastity?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/12-the-role-of-women?lang=eng", "https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/12-the-role-of-women?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/13-family?lang=eng", "https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/13-family?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/14-holy-places?lang=eng", "https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/14-holy-places?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/15-life-after-death?lang=eng", "https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/15-life-after-death?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/17-religious-diversity?lang=eng", "https://www.churchofjesuschrist.org/study/manual/muslims-and-latter-day-saints/17-religious-diversity?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/scriptures/the-living-christ-the-testimony-of-the-apostles/the-living-christ-the-testimony-of-the-apostles?lang=eng", "https://www.churchofjesuschrist.org/study/scriptures/the-living-christ-the-testimony-of-the-apostles/the-living-christ-the-testimony-of-the-apostles?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/scriptures/the-family-a-proclamation-to-the-world/the-family-a-proclamation-to-the-world?lang=eng", "https://www.churchofjesuschrist.org/study/scriptures/the-family-a-proclamation-to-the-world/the-family-a-proclamation-to-the-world?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/scriptures/the-restoration-of-the-fulness-of-the-gospel-of-jesus-christ/a-bicentennial-proclamation-to-the-world?lang=eng", "https://www.churchofjesuschrist.org/study/scriptures/the-restoration-of-the-fulness-of-the-gospel-of-jesus-christ/a-bicentennial-proclamation-to-the-world?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2024/12/11runia?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2024/12/11runia?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2024/12/12palmer?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2024/12/12palmer?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2024/12/13cook?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2024/12/13cook?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2024/12/14oaks?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2024/12/14oaks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2023/12/11browning?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2023/12/11browning?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2023/12/12johnson?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2023/12/12johnson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2023/12/13gong?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2023/12/13gong?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2023/12/14nelson?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2023/12/14nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2022/12/11cordon?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2022/12/11cordon?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2022/12/12teixeira?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2022/12/12teixeira?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2022/12/13andersen?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2022/12/13andersen?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2022/12/14oaks?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2022/12/14oaks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2021/12-01/11craig?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2021/12-01/11craig?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2021/12-01/12bassett?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2021/12-01/12bassett?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2021/12-01/13renlund?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2021/12-01/13renlund?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2021/12-01/14eyring?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2021/12-01/14eyring?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2020/12/15nelson?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2020/12/15nelson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2020/12/13holland?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2020/12/13holland?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2020/12/12nielson?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2020/12/12nielson?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2019/12/14oaks?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2019/12/14oaks?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2018/12/silent-night-loves-pure-light?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2018/12/silent-night-loves-pure-light?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2018/12/a-gift-from-father-accepted-or-rejected?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2018/12/a-gift-from-father-accepted-or-rejected?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2018/12/let-every-heart-prepare-him-room?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2018/12/let-every-heart-prepare-him-room?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2018/12/four-gifts-that-jesus-christ-offers-to-you?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2018/12/four-gifts-that-jesus-christ-offers-to-you?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2017/12/heavenly-gifts?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2017/12/heavenly-gifts?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2017/12/scatter-your-crumbs?lang=eng", "https://www.churchofjesuschrist.org/study/broadcasts/christmas-devotional/2017/12/scatter-your-crumbs?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/liahona/messages/2025/10/03-covenants-power-and-promises?lang=eng", "https://www.churchofjesuschrist.org/study/liahona/messages/2025/10/03-covenants-power-and-promises?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/liahona/messages/2025/09/03-blessed-by-priesthood-authority-and-power?lang=eng", "https://www.churchofjesuschrist.org/study/liahona/messages/2025/09/03-blessed-by-priesthood-authority-and-power?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/liahona/messages/2025/08/03-the-truth-of-our-lives?lang=eng", "https://www.churchofjesuschrist.org/study/liahona/messages/2025/08/03-the-truth-of-our-lives?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/liahona/messages/2025/07/03-find-joy-in-your-gospel-journey?lang=eng", "https://www.churchofjesuschrist.org/study/liahona/messages/2025/07/03-find-joy-in-your-gospel-journey?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/liahona/messages/2025/06/03-we-follow-jesus-christ?lang=eng", "https://www.churchofjesuschrist.org/study/liahona/messages/2025/06/03-we-follow-jesus-christ?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/liahona/messages/2025/04/03-jesus-christ-the-hope-and-promise-of-easter?lang=eng", "https://www.churchofjesuschrist.org/study/liahona/messages/2025/04/03-jesus-christ-the-hope-and-promise-of-easter?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/liahona/messages/2025/03/03-heavenly-father-wants-to-speak-to-you?lang=eng", "https://www.churchofjesuschrist.org/study/liahona/messages/2025/03/03-heavenly-father-wants-to-speak-to-you?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/liahona/messages/2025/02/03-i-will-heal-them?lang=eng", "https://www.churchofjesuschrist.org/study/liahona/messages/2025/02/03-i-will-heal-them?lang=pes"),
        ("https://www.churchofjesuschrist.org/study/liahona/messages/2025/01/04-glad-tidings-of-love-and-joy?lang=eng", "https://www.churchofjesuschrist.org/study/liahona/messages/2025/01/04-glad-tidings-of-love-and-joy?lang=pes")
    ]

    # Clear the file before starting a new run
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        pass

    for index, (en_url, fa_url) in enumerate(talk_pairs):
        scrape_and_process_pair_strict(index, en_url, fa_url, CONTENT_SELECTOR, OUTPUT_FILE, PARAGRAPH_TAG)

Talk #: 0
Number of English Paragraphs: 43
Number of Persian Paragraphs: 43
✅ Processed https://www.churchofjesuschrist.org/study/general-conference/2025/10/12stevenson?lang=eng. Saved 101 pairs. Discarded 4 misaligned/empty paragraphs.
Talk #: 1
Number of English Paragraphs: 33
Number of Persian Paragraphs: 33
✅ Processed https://www.churchofjesuschrist.org/study/general-conference/2025/10/13browning?lang=eng. Saved 68 pairs. Discarded 2 misaligned/empty paragraphs.
Talk #: 2
Number of English Paragraphs: 22
Number of Persian Paragraphs: 22
✅ Processed https://www.churchofjesuschrist.org/study/general-conference/2025/10/14barcellos?lang=eng. Saved 53 pairs. Discarded 4 misaligned/empty paragraphs.
Talk #: 3
Number of English Paragraphs: 25
Number of Persian Paragraphs: 25
✅ Processed https://www.churchofjesuschrist.org/study/general-conference/2025/10/15eyre?lang=eng. Saved 61 pairs. Discarded 4 misaligned/empty paragraphs.
Talk #: 4
Number of English Paragraphs: 19
Number of Persian 

## 3. Build the Hugging Face Dataset

Load the scraped JSONL, reshape it into the `{"translation": {"en": ..., "fa": ...}}`
format expected by seq2seq tooling, and split into train/test.


In [44]:
from datasets import load_dataset, Dataset

raw_datasets = load_dataset('json', data_files='DATASET.jsonl', split='train')

def create_translation_dict(example):
    return {
        'translation': {
            'en': example['english'],
            'fa': example['persian']
        }
    }

transformed_datasets = raw_datasets.map(create_translation_dict)

final_datasets = transformed_datasets.remove_columns(['english', 'persian'])

train_test_split = final_datasets.train_test_split(test_size=0.1)

train = train_test_split['train']
test = train_test_split['test']

print(train_test_split)
# DatasetDict({
#     train: Dataset(...)
#     test: Dataset(...)
# })
print(train_test_split['train'][0])
# {'translation': {'en': '...', 'fa': '...'}}

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 43952
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 4884
    })
})
{'translation': {'en': 'This woodpile contains an enormous amount of fuel, capable of producing light and heat for days.', 'fa': 'این تودۀ عظیم چوبی دارای سوختی است که قادر به ایجاد حرارت و نور عظیمی برای چندین روز میباشد.'}}


## 4. Load Base Model & Tokenizer

In [45]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

model_name = "facebook/mbart-large-50-many-to-many-mmt"

tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

### Sanity check: translate with the *un-fine-tuned* base model

In [46]:
text = "God is merciful and full of compassion."

# Prepare inputs
inputs = tokenizer(text, return_tensors="pt")

# Generate translation, forcing Persian as output language
generated_tokens = model.generate(
    **inputs,
    forced_bos_token_id=tokenizer.lang_code_to_id["fa_IR"]
)

persian = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)
print(persian)

خداوند بخشنده و پر از شفقت است.


## 5. Fine-Tune

Tokenize the dataset, then fine-tune with `Seq2SeqTrainer`.

Key settings: effective batch size 8 (batch size 1 × 8 gradient accumulation
steps), learning rate 2e-5, 3 epochs, fp16, label smoothing 0.1 — conservative
choices suited to fine-tuning a large pretrained model on a single Colab GPU.


In [68]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

# Set source and target languages on the tokenizer
tokenizer.src_lang = "en_XX"
tokenizer.tgt_lang = "fa_IR"


def preprocess(batch):
    src = [x["en"] for x in batch["translation"]]
    tgt = [x["fa"] for x in batch["translation"]]

    model_inputs = tokenizer(
        src, text_target=tgt, max_length=256, truncation=True
    )

    return model_inputs


train_tokenized = train.map(
    preprocess, batched=True, remove_columns=train.column_names
)
test_tokenized = test.map(
    preprocess, batched=True, remove_columns=test.column_names
)

# Estimate total training steps
effective_batch_size = 4 * 8  # per_device_train_batch_size * gradient_accumulation_steps
steps_per_epoch = max(1, len(train_tokenized) // effective_batch_size)
total_steps = int(steps_per_epoch * 3)
print(
    f"Effective batch size: {effective_batch_size} | Steps/epoch:"
    f" {steps_per_epoch} | Estimated total steps: {total_steps}"
)

args = Seq2SeqTrainingArguments(
    output_dir="mbart-fa-religious-final",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    num_train_epochs=3,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    predict_with_generate=True,
    generation_num_beams=4,
    remove_unused_columns=True,
    label_smoothing_factor=0,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, label_pad_token_id=-100
)

fa_id = tokenizer.convert_tokens_to_ids("fa_IR")
model.generation_config.decoder_start_token_id = tokenizer.eos_token_id  # 2
model.generation_config.forced_bos_token_id = tokenizer.lang_code_to_id["fa_IR"]

model.config.forced_bos_token_id = None

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=data_collator,
    processing_class=tokenizer,
)

Map:   0%|          | 0/43952 [00:00<?, ? examples/s]

Map:   0%|          | 0/4884 [00:00<?, ? examples/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Effective batch size: 32 | Steps/epoch: 1373 | Estimated total steps: 4119


In [54]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,10.218759,1.240696
2,9.414924,1.177699
3,7.799063,1.171228


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=4122, training_loss=9.46598117088936, metrics={'train_runtime': 3637.1872, 'train_samples_per_second': 36.252, 'train_steps_per_second': 1.133, 'total_flos': 1.2127562933993472e+16, 'train_loss': 9.46598117088936, 'epoch': 3.0})

### Confirm the metrics of the saved (best) checkpoint

`load_best_model_at_end=True` means the trainer has already swapped in whichever evaluated checkpoint had the lowest `eval_loss`. This just prints its metrics clearly rather than digging them out of the logs.


In [56]:
final_metrics = trainer.evaluate()
print(final_metrics)


Training Loss,Validation Loss,Epoch
7.799063,1.171228,3


{'eval_loss': 1.171228051185608}


## 6. Save the Model

Save to Google Drive (persistent storage across Colab sessions) and push to
the Hugging Face Hub.


In [57]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [58]:
from huggingface_hub import notebook_login
notebook_login()

In [69]:
# Define your permanent save directory
save_directory = "/content/drive/MyDrive/Models/mbart-fa-religious-final"

# Save the model and tokenizer to Google Drive
trainer.save_model(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"Model successfully saved to: {save_directory}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model successfully saved to: /content/drive/MyDrive/Models/mbart-fa-religious-final


In [70]:
trainer.push_to_hub("mbart-fa-religious-final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/tuckj90/mbart-fa-religious-final/commit/1ef3411caa01ef7b9763f293c03d0256f4c65fc9', commit_message='mbart-fa-religious-final', commit_description='', oid='1ef3411caa01ef7b9763f293c03d0256f4c65fc9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/tuckj90/mbart-fa-religious-final', endpoint='https://huggingface.co', repo_type='model', repo_id='tuckj90/mbart-fa-religious-final'), pr_revision=None, pr_num=None)

## 7. Inference & Evaluation

Reload the fine-tuned model from the Hub and translate held-out test examples.


In [71]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

tokenizer = MBart50TokenizerFast.from_pretrained(
    "mbart-fa-religious-final",
    src_lang="en_XX"
)

model = MBartForConditionalGeneration.from_pretrained(
    "mbart-fa-religious-final"
).to("cuda")

tokenizer.tgt_lang = "fa_IR"


Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

In [72]:
def translate(text, num_beams=4):
    encoded = tokenizer(text, return_tensors="pt").to("cuda")

    generated_tokens = model.generate(
        **encoded,
        forced_bos_token_id=tokenizer.lang_code_to_id["fa_IR"],
        num_beams=num_beams,
    )

    return tokenizer.decode(generated_tokens[0], skip_special_tokens=True)


In [76]:
translate("God said, let there be light")


'خدا گفت، بگذار نور باشد'

### BLEU on the held-out test set

In [77]:
from evaluate import load

bleu = load("bleu")
results = bleu.compute(
    predictions=[translate(x["translation"]["en"]) for x in test],
    references=[[x["translation"]["fa"]] for x in test]
)
print(results)

{'bleu': 0.26273285003844005, 'precisions': [0.5806799057556379, 0.3266162803604404, 0.19947358925103212, 0.12594979912937412], 'brevity_penalty': 1.0, 'length_ratio': 1.0464636502697044, 'translation_length': 103985, 'reference_length': 99368}
